# 🌳 Módulo 7: The Terminal Playground — CLI Tools
## Notebook de Conocimiento Guiado

**Objetivo:** Dominar la creación de herramientas CLI: streams Unix, ANSI escape codes,
generación procedural de arte ASCII y scraping del sistema.

**Lore:** Eres el Mago del Terminal Intergaláctico. Con código y ASCII, transformas
la oscura terminal en una obra de arte viviente.

| Sección | Concepto |
|---------|---------|
| 📚 Teoría | stdin/stdout/stderr, pipes, señales |
| 🔨 Demo | ANSI escape codes en Python |
| 🔨 Demo | Neofetch clone con /proc |
| ✏️ Ejercicio 1 | Banner de colores ANSI cíclicos |
| ✏️ Ejercicio 2 | Mini bonsai ASCII generativo |
| 🎯 Quiz | Unix pipes, streams, señales |

## 📚 Parte 1: Filosofía Unix — Todo es un Stream

```
stdin  (fd=0): entrada  → el teclado o la salida de otro proceso
stdout (fd=1): salida   → la terminal o la entrada de otro proceso  
stderr (fd=2): errores  → siempre a la terminal (no se pipelea por defecto)
```

**Pipes** conectan stdout de un proceso con stdin del siguiente:
```bash
$ cat archivo.txt | grep "error" | sort | uniq -c | sort -rn
         ↑               ↑            ↑        ↑           ↑
      produce         filtra       ordena  desdup       reordena
```

**¿Por qué los pipes son poderosos?**
- Cada herramienta hace UNA cosa bien (principio Unix)
- Se pueden combinar sin límite
- Procesan datos en streaming (no cargan todo en RAM)

### ANSI Escape Codes
Secuencias especiales que el terminal interpreta como comandos:
```
\033[31m    → color rojo
\033[1m     → negrita
\033[0m     → reset
\033[2J     → limpiar pantalla
\033[H      → mover cursor al inicio
```

In [ ]:
# 🔨 ANSI Escape Codes en Python

# Colores de texto (foreground)
COLORES = {
    'negro': '\033[30m',   'rojo':     '\033[31m',
    'verde': '\033[32m',   'amarillo': '\033[33m',
    'azul':  '\033[34m',   'magenta':  '\033[35m',
    'cyan':  '\033[36m',   'blanco':   '\033[37m',
}
# Estilos
NEGRITA  = '\033[1m'
SUBRAY   = '\033[4m'
RESET    = '\033[0m'

def colorear(texto, color='blanco', negrita=False, subrayado=False):
    código = ''
    if negrita:     código += NEGRITA
    if subrayado:   código += SUBRAY
    código += COLORES.get(color, '')
    return f"{código}{texto}{RESET}"

# Demo de colores
print("Paleta de colores ANSI:")
for nombre, _ in COLORES.items():
    print(f"  {colorear(f'■ {nombre}', nombre, negrita=True)}")

print()
print(colorear("¡Bienvenido al Terminal Playground!", 'cyan', negrita=True))
print(colorear("ERROR: archivo no encontrado", 'rojo'))
print(colorear("OK: operación exitosa", 'verde'))

In [ ]:
# 🔨 IMPLEMENTACIÓN: Neofetch Clone

import platform
import os
import socket
import time

def leer_archivo_proc(path, default="desconocido"):
    """Lee un archivo /proc en Linux de forma segura."""
    try:
        with open(path, 'r') as f:
            return f.read()
    except (IOError, OSError):
        return default

def neofetch_info():
    """Recopila información del sistema como neofetch."""
    info = {}
    
    # Sistema operativo
    info['os'] = platform.system() + " " + platform.release()
    info['arquitectura'] = platform.machine()
    info['hostname'] = socket.gethostname()
    info['python'] = platform.python_version()
    
    # CPU (desde /proc/cpuinfo en Linux, o platform en otros)
    cpu_raw = leer_archivo_proc('/proc/cpuinfo')
    if cpu_raw != "desconocido":
        for line in cpu_raw.split('\n'):
            if 'model name' in line:
                info['cpu'] = line.split(':')[1].strip()
                break
    else:
        info['cpu'] = platform.processor() or "desconocido"
    
    # RAM (desde /proc/meminfo)
    mem_raw = leer_archivo_proc('/proc/meminfo')
    if mem_raw != "desconocido":
        for line in mem_raw.split('\n'):
            if line.startswith('MemTotal'):
                kb = int(line.split()[1])
                info['ram'] = f"{kb // 1024} MB"
                break
    
    # Uptime
    uptime_raw = leer_archivo_proc('/proc/uptime', "0")
    if uptime_raw != "0":
        segundos = float(uptime_raw.split()[0])
        horas = int(segundos // 3600)
        minutos = int((segundos % 3600) // 60)
        info['uptime'] = f"{horas}h {minutos}m"
    
    return info

LOGO_ASCII = """
    .-.
   (   )
    '-'
   //|\\
  // | \\
"""

info = neofetch_info()
print(colorear("╔══════════════════════════════╗", 'cyan'))
print(colorear("║    🖥️  SISTEMA DE INFORMACIÓN    ║", 'cyan', negrita=True))
print(colorear("╚══════════════════════════════╝", 'cyan'))
for clave, valor in info.items():
    print(f"  {colorear(clave.upper() + ':', 'amarillo', negrita=True)} {valor}")

## 🏗️ Parte 3: Generación Procedural — Bonsai ASCII

El generador de bonsai usa un **random walk** recursivo:
- El tronco crece hacia arriba
- En cada paso, puede: continuar recto, doblar izquierda, doblar derecha, o bifurcarse
- Las ramas se vuelven más delgadas con la profundidad

In [ ]:
# 🔨 IMPLEMENTACIÓN: Mini Bonsai Generativo

import random

def generar_bonsai(semilla=42, ancho=60, alto=25):
    """Genera un bonsai ASCII con random walk."""
    random.seed(semilla)
    canvas = [[' '] * ancho for _ in range(alto)]
    
    def plantar(x, y, dx, dy, energia, grosor):
        """Recursivamente crece una rama."""
        if energia <= 0 or y < 0 or y >= alto or x < 0 or x >= ancho:
            return
        
        # Elegir carácter según grosor
        if grosor > 3:
            char = '|' if abs(dy) > abs(dx) else '-'
        elif grosor > 1:
            char = random.choice(['/', '\\', '|'])
        else:
            char = random.choice(['&', '*', '.', ','])  # hojas
        
        canvas[y][x] = char
        
        # Próximo paso
        prob = random.random()
        nueva_energia = energia - 1
        
        if prob < 0.1 and energia > 5:
            # Bifurcación: crecer en dos direcciones
            plantar(x - 1, y - 1, -1, -1, nueva_energia - 2, grosor - 1)
            plantar(x + 1, y - 1, +1, -1, nueva_energia - 2, grosor - 1)
        elif prob < 0.4:
            # Doblar izquierda
            plantar(x - 1, y - 1, -1, dy, nueva_energia, max(1, grosor - 1))
        elif prob < 0.7:
            # Doblar derecha
            plantar(x + 1, y - 1, +1, dy, nueva_energia, max(1, grosor - 1))
        else:
            # Continuar recto
            plantar(x + dx, y - 1, dx, dy, nueva_energia, grosor)
    
    # Base del árbol
    base_x = ancho // 2
    base_y = alto - 1
    
    # Dibujar tronco base
    for i in range(4):
        canvas[base_y - i][base_x] = '|'
    
    # Crecer ramas principales
    plantar(base_x - 1, base_y - 4, -1, -1, energia=15, grosor=4)
    plantar(base_x + 1, base_y - 4, +1, -1, energia=15, grosor=4)
    
    # Maceta
    for dx in range(-3, 4):
        canvas[alto - 1][base_x + dx] = '_'
    canvas[alto - 1][base_x - 3] = '('
    canvas[alto - 1][base_x + 3] = ')'
    
    return canvas

bonsai = generar_bonsai(semilla=7)
print(colorear("🌳 Bonsai Galáctico", 'verde', negrita=True))
for fila in bonsai:
    línea = ''.join(fila)
    if any(c in línea for c in ['&', '*', '.', ',']):
        print(colorear(línea, 'verde'))
    elif any(c in línea for c in ['|', '/', '\\']):
        print(colorear(línea, 'amarillo'))
    else:
        print(línea)

## ✏️ Ejercicio 1: Banner con colores ANSI cíclicos

**Problema:** Implementa `banner_arcoiris(texto)` que imprime el texto con
cada carácter en un color diferente, ciclando por todos los colores ANSI.

In [ ]:
# ✏️ EJERCICIO 1

def banner_arcoiris(texto):
    """Imprime texto con colores ANSI cíclicos por carácter."""
    # TODO: Implementar
    # colores disponibles: ['rojo', 'amarillo', 'verde', 'cyan', 'azul', 'magenta']
    pass

banner_arcoiris("¡Hola Mundo Galáctico!")

In [ ]:
# 💡 SOLUCIÓN — Ejercicio 1

def banner_arcoiris(texto):
    ciclo_colores = ['rojo', 'amarillo', 'verde', 'cyan', 'azul', 'magenta']
    resultado = ""
    i = 0
    for char in texto:
        if char != ' ':
            resultado += colorear(char, ciclo_colores[i % len(ciclo_colores)], negrita=True)
            i += 1
        else:
            resultado += ' '
    print(resultado)

banner_arcoiris("¡Hola Mundo Galáctico!")
banner_arcoiris("The Terminal Playground")

## ✏️ Ejercicio 2: Mini bonsai con posición controlada

In [ ]:
# ✏️ EJERCICIO 2

# TODO: Modifica generar_bonsai para aceptar un parámetro 'inclinacion'
# Si inclinacion = 'izquierda': la mayoría de ramas van a la izquierda
# Si inclinacion = 'derecha': la mayoría van a la derecha
# Si inclinacion = 'simetrico': igual de ambos lados

def bonsai_con_inclinacion(inclinacion='simetrico', semilla=42):
    # Tu código aquí...
    pass

bonsai_con_inclinacion('izquierda')

In [ ]:
# 💡 SOLUCIÓN — Ejercicio 2

def bonsai_con_inclinacion(inclinacion='simetrico', semilla=42):
    random.seed(semilla)
    ancho, alto = 60, 20
    canvas = [[' '] * ancho for _ in range(alto)]
    
    def crecer(x, y, dx, energia, grosor):
        if energia <= 0 or y < 1 or x < 1 or x >= ancho - 1: return
        char = '|' if grosor > 2 else random.choice(['/', '\\', '*', '.'])
        canvas[y][x] = char
        
        # Pesos según inclinación
        if inclinacion == 'izquierda':
            pesos = [0.5, 0.3, 0.2]   # izq, recto, der
        elif inclinacion == 'derecha':
            pesos = [0.2, 0.3, 0.5]
        else:
            pesos = [0.33, 0.34, 0.33]
        
        r = random.random()
        if r < pesos[0]:
            crecer(x - 1, y - 1, -1, energia - 1, max(1, grosor - 1))
        elif r < pesos[0] + pesos[1]:
            crecer(x,     y - 1,  0, energia - 1, grosor)
        else:
            crecer(x + 1, y - 1, +1, energia - 1, max(1, grosor - 1))
        
        if energia > 8 and random.random() < 0.3:
            crecer(x + (-1 if dx == 1 else 1), y - 1, -dx, energia - 3, max(1, grosor - 2))
    
    base_x = ancho // 2
    for i in range(3): canvas[alto - 1 - i][base_x] = '|'
    crecer(base_x, alto - 4, 0, energia=14, grosor=4)
    
    print(f"Bonsai inclinado hacia: {inclinacion}")
    for fila in canvas:
        print(''.join(fila))

bonsai_con_inclinacion('izquierda', semilla=7)

## 🎯 Quiz — Preguntas de Entrevista

**P1:** ¿Cuál es la diferencia entre stdout y stderr en Unix?
> **R:** stdout (fd=1) es la salida normal del programa. stderr (fd=2) es para errores y diagnósticos.
> Los pipes (`|`) solo transfieren stdout por defecto. Para redirigir stderr: `2>&1` o `2>archivo`.

**P2:** ¿Cómo evitarías buffering en un programa Python que hace pipe?
> **R:** `python -u script.py` desactiva el buffering. También se puede usar
> `sys.stdout.flush()` después de cada print, o `print(x, flush=True)`.

**P3:** ¿Qué señal se envía al presionar Ctrl+C y cómo se maneja en Python?
> **R:** SIGINT. En Python se convierte en `KeyboardInterrupt`. Se puede capturar con
> `signal.signal(signal.SIGINT, handler)` o con `try/except KeyboardInterrupt`.

**P4:** ¿Por qué no deberías parsear la salida de `ls`?
> **R:** Los nombres de archivos pueden contener espacios, saltos de línea, caracteres especiales.
> Usar `os.listdir()`, `pathlib.Path.iterdir()`, o `find ... -print0 | xargs -0` en shell.